In [8]:
# ==============================================================================
# Atividade 2: Chatbot Versão 2 (Decision Tree) com Fallback
# ==============================================================================

import os
import random
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

CSV_PATH = 'dataset_moveis_100.csv'

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': ['quero comprar', 'qual o preco do', 'tem cupom para', 'como faco para adquirir', 'desejo orcamento de'],
        'o': ['sofa retratil 3 lugares', 'conjunto de mesa de jantar', 'guarda roupa casal', 'painel para tv', 'colchao queen size']
    },
    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': ['como montar o', 'onde baixo o manual do', 'estou com duvida no', 'veio faltando parafuso no', 'preciso de assistencia para'],
        'o': ['armario de cozinha', 'rack da sala', 'berco do bebe', 'esquema de montagem', 'manual da estante']
    },
    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': ['preciso trocar o', 'quero devolver a', 'como solicito o estorno do', 'desejo solicitar a troca da', 'como funciona a devolucao do'],
        'o': ['produto com defeito', 'mesa que veio arranhada', 'cadeira no prazo de 7 dias', 'pedido cancelado', 'item com avaria']
    },
    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': ['estou indignado com o', 'quero fazer uma queixa do', 'estou reclamando do', 'produto veio quebrado e o', 'atendimento horrivel do'],
        'o': ['atraso na minha entrega', 'servico de montagem', 'sac que nao responde', 'pos venda da loja', 'estado do meu movel']
    },
    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': ['onde esta o meu', 'qual o prazo de entrega do', 'como rastreio a', 'qual a transportadora do', 'quando chega o'],
        'o': ['meu pedido', 'codigo de rastreamento', 'movel comprado', 'status do envio', 'agendamento da entrega']
    }
}

amostras = []
random.seed(42)

for intencao, comp in templates.items():
    for _ in range(20):
      s = random.choice(comp['s'])
      a = random.choice(comp['a'])
      o = random.choice(comp['o'])
      frase = f"{s} {a} {o}".strip().capitalize()
      amostras.append({'texto': frase, 'intencao': intencao})

df_moveis = pd.DataFrame(amostras)
df_moveis.to_csv(CSV_PATH, index=False, encoding='utf-8')
print(f"Dataset '{CSV_PATH}' criado com sucesso!\n")

# 1. Carregar o dataset
df = pd.read_csv(CSV_PATH)

# 2. Divisão estratificada treino/teste (30% teste)
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

# 3. Pipeline: TF-IDF + Árvore de Decisão
pipeline_tree = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# 4. Treinamento
pipeline_tree.fit(X_train, y_train)

# 5. Avaliação no conjunto de teste
y_pred = pipeline_tree.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print("="*60)
print("MÉTRICAS GERAIS DO MODELO (Decision Tree)")
print(f"Acurácia Geral: {acc*100:.2f}%")
print(f"F1-Score Geral (Weighted): {f1*100:.2f}%")
print("="*60)

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

print("=== MATRIZ DE CONFUSÃO ===")
labels = sorted(df['intencao'].unique())
print("Ordem das classes:", labels)
print(confusion_matrix(y_test, y_pred, labels=labels))

# 6. Bateria de 8 testes manuais com fallback
LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===")

for i in range(1, 9):
    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()

    # Árvore de decisão (predict_proba)
    probs = pipeline_tree.predict_proba([frase])[0]
    maior_prob = np.max(probs)
    intencao = pipeline_tree.predict([frase])[0]

    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Bot: Intenção identificada -> {intencao.upper()} "
              f"(Confiança: {maior_prob*100:.1f}%)")
    else:
        print(f"Bot: [FALLBACK - Confiança baixa: {maior_prob*100:.1f}%]")
        print("Desculpe, não entendi sua solicitação. "
              "Encaminhando você para um atendente humano...")

Dataset 'dataset_moveis_100.csv' criado com sucesso!

MÉTRICAS GERAIS DO MODELO (Decision Tree)
Acurácia Geral: 80.00%
F1-Score Geral (Weighted): 79.74%

=== RELATÓRIO DE CLASSIFICAÇÃO ===
                    precision    recall  f1-score   support

logistica_entregas       0.80      0.67      0.73         6
       reclamacoes       1.00      0.67      0.80         6
           suporte       0.75      1.00      0.86         6
 trocas_devolucoes       0.83      0.83      0.83         6
            vendas       0.71      0.83      0.77         6

          accuracy                           0.80        30
         macro avg       0.82      0.80      0.80        30
      weighted avg       0.82      0.80      0.80        30

=== MATRIZ DE CONFUSÃO ===
Ordem das classes: ['logistica_entregas', 'reclamacoes', 'suporte', 'trocas_devolucoes', 'vendas']
[[4 0 0 0 2]
 [1 4 1 0 0]
 [0 0 6 0 0]
 [0 0 1 5 0]
 [0 0 0 1 5]]

=== INICIANDO BATERIA DE TESTES (8 INPUTS OBRIGATÓRIOS) ===

[Teste 1/8]
Di